# Sinh biểu đồ coherence / diversity / runtime / robustness — CafeBERT full benchmark

Notebook này đọc trực tiếp `benchmark/cafebert_full/reference/full_results.csv` (480 dòng thật: 4 corpus × 6 mô hình × 5 giá trị k × 4 seed, đã audit `PASS`) và vẽ 4 nhóm biểu đồ, lưới 2×2 theo 4 corpus tiếng Việt (benchmark chỉ dùng một encoder CafeBERT nên không có chiều encoder).

Cột dữ liệu: **Coherence** → `wec_in` · **Diversity** → `topic_diversity` · **Runtime** → `fit_seconds`/`pipeline_seconds` (KHÔNG gộp lẫn) · **Robustness** → `c_npmi` (chỉ số phụ).

Phần cuối notebook (**Concept Compass**) không đọc từ CSV mà load một checkpoint S³ thật trong `artifacts/turftopic/…/models/` — cần chạy `make train-news` (hoặc `train-visfd`) trước.

**Chạy notebook**: cần `jupyter`/`ipykernel` trong `.venv`:
```powershell
.venv\Scripts\python.exe -m pip install jupyter ipykernel
```
Biểu đồ xuất ra `benchmark/cafebert_full/notebook_charts/`.

In [ ]:
from __future__ import annotations

from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd


def find_repo_root(marker: str = "benchmark/cafebert_full/reference/full_results.csv") -> Path:
    """Works whether the notebook's cwd is the repo root or benchmark/cafebert_full/."""
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / marker).exists():
            return candidate
    raise FileNotFoundError(
        f"Khong tim thay '{marker}' tu {Path.cwd()} hay bat ky thu muc cha nao. "
        "Hay chay notebook nay tu repo root (E:/Development/NLP_S3) hoac tu benchmark/cafebert_full/."
    )


ROOT = find_repo_root()
REFERENCE_DIR = ROOT / "benchmark" / "cafebert_full" / "reference"
CSV_PATH = REFERENCE_DIR / "full_results.csv"
OUTPUT_DIR = ROOT / "benchmark" / "cafebert_full" / "notebook_charts"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Repo root: {ROOT}")
print(f"Doc du lieu tu: {CSV_PATH}")
print(f"Xuat bieu do vao: {OUTPUT_DIR}")

In [ ]:
# Cung tu vung mau/nhan voi benchmark/cafebert_full/generate_cafebert_full_report.py.

TOPIC_COUNTS = [10, 20, 30, 40, 50]
SEEDS = [11, 29, 42, 47]

MODEL_ORDER = ["s3_axial", "s3_angular", "s3_combined", "lda", "nmf", "bertopic_kmeans"]
MODEL_LABELS = {
    "s3_axial": "S\u00b3 axial",
    "s3_angular": "S\u00b3 angular",
    "s3_combined": "S\u00b3 combined",
    "lda": "LDA",
    "nmf": "NMF",
    "bertopic_kmeans": "BERTopic + UMAP + KMeans",
}
# Xanh duong/tim/xanh la = 3 bien the S3; nau/xam/do = 3 baseline. TAT CA cac
# duong ve dam, ro rang -- phan biet S3 vs baseline bang do day net + zorder.
COLORS = {
    "S\u00b3 axial": "#1d4ed8",
    "S\u00b3 angular": "#7c3aed",
    "S\u00b3 combined": "#15803d",
    "LDA": "#b45309",
    "NMF": "#475569",
    "BERTopic + UMAP + KMeans": "#dc2626",
}

CORPUS_ORDER = ["vietnamese-news", "visfd", "vi-medical", "vntc-it"]
CORPUS_LABELS = {
    "vietnamese-news": "Vietnamese-news",
    "visfd": "UIT-ViSFD",
    "vi-medical": "ViMedical Disease",
    "vntc-it": "VNTC-CNTT",
}

In [ ]:
frame = pd.read_csv(CSV_PATH)
frame["method"] = pd.Categorical(
    frame["model"].map(MODEL_LABELS), [MODEL_LABELS[m] for m in MODEL_ORDER], ordered=True
)
frame["corpus_label"] = pd.Categorical(
    frame["corpus"].map(CORPUS_LABELS), [CORPUS_LABELS[c] for c in CORPUS_ORDER], ordered=True
)

expected_rows = len(CORPUS_ORDER) * len(MODEL_ORDER) * len(SEEDS) * len(TOPIC_COUNTS)
assert len(frame) == expected_rows, f"Ky vong {expected_rows} dong, doc duoc {len(frame)}"
assert frame["status"].eq("ok").all(), "Co dong status != 'ok', kiem tra lai truoc khi ve bieu do"
print(f"OK: {len(frame)} dong, tat ca status=ok.")
frame.head(3)

## Hàm vẽ dùng chung

Tất cả đường vẽ **đậm, rõ nét**; phân biệt S³ với baseline bằng độ dày nét (S³ `linewidth=3.2`, baseline `2.2`) + `zorder` (S³ vẽ đè lên trên). Marker S³ to hơn (`6.5` vs `5.5`).

**Cỡ chữ**: `base_fontsize=26` (gấp ~2 lần mặc định matplotlib) — tiêu đề ô `28`, suptitle `30`, tick `25`. Figure to tương ứng (`20×14`, dpi 180).

In [ ]:
def plot_metric_grid(
    frame: pd.DataFrame,
    metric: str,
    ylabel: str,
    title: str,
    output_path: Path,
    log_scale: bool = False,
    s3_models: list[str] | None = None,
    s3_label_override: str | None = None,
    s3_linewidth: float = 3.4,
    baseline_linewidth: float = 2.4,
    s3_line_alpha: float = 1.0,
    baseline_line_alpha: float = 0.95,
    s3_fill_alpha: float = 0.14,
    baseline_fill_alpha: float = 0.07,
    base_fontsize: int = 26,
) -> Path:
    """2x2 grid (1 subplot / corpus), 1 duong mau / mo hinh, dai mean+-SD.
    Moi duong deu dam; S3 day hon + ve len tren. Font gap ~2 lan mac dinh.

    s3_models: mac dinh ca 3 bien the (axial/angular/combined) -- dung cho
    coherence/diversity/robustness, noi 3 cong thuc beta cho diem khac nhau
    that. Voi runtime (fit_seconds/pipeline_seconds), ca 3 bien the deu suy
    tu CUNG mot lan fit/refit da cache nen gan nhu trung khop tuyet doi --
    ve ca 3 se khien 2 duong bi duong con lai ve sau de kin (khong phai loi
    thieu du lieu). Truyen s3_models=["s3_combined"] (+ s3_label_override)
    de chi ve 1 duong dai dien, tranh nham la "mat line"."""
    s3_models = s3_models if s3_models is not None else [m for m in MODEL_ORDER if m.startswith("s3_")]
    baseline_models = [m for m in MODEL_ORDER if not m.startswith("s3_")]
    models_to_draw = baseline_models + s3_models
    label_of = dict(MODEL_LABELS)
    if s3_label_override and len(s3_models) == 1:
        label_of[s3_models[0]] = s3_label_override

    plt.style.use("seaborn-v0_8-whitegrid")
    plt.rcParams.update({"font.size": base_fontsize})
    fig, axes = plt.subplots(2, 2, figsize=(20, 14), dpi=180, sharex=True)

    for ax, corpus in zip(axes.flat, CORPUS_ORDER, strict=True):
        part = frame.loc[frame["corpus"] == corpus].copy()
        part["model_label"] = part["model"].map(label_of)
        aggregate = (
            part.groupby(["model_label", "n_topics"], observed=True)[metric]
            .agg(["mean", "std"])
            .reset_index()
        )
        draw_order = baseline_models + s3_models
        for model in draw_order:
            method = label_of[model]
            values = aggregate.loc[aggregate["model_label"] == method].sort_values("n_topics")
            is_s3 = model.startswith("s3_")
            line_alpha = s3_line_alpha if is_s3 else baseline_line_alpha
            fill_alpha = s3_fill_alpha if is_s3 else baseline_fill_alpha
            color = COLORS[MODEL_LABELS[model]]
            ax.plot(
                values["n_topics"],
                values["mean"],
                color=color,
                alpha=line_alpha,
                marker="o",
                markersize=8 if is_s3 else 6.5,
                linewidth=s3_linewidth if is_s3 else baseline_linewidth,
                label=method,
                zorder=3 if is_s3 else 2,
            )
            sd = values["std"].fillna(0)
            ax.fill_between(
                values["n_topics"],
                values["mean"] - sd,
                values["mean"] + sd,
                color=color,
                alpha=fill_alpha,
                zorder=1,
            )
        if log_scale:
            ax.set_yscale("log")
        ax.set_title(CORPUS_LABELS[corpus], loc="left", fontweight="bold", fontsize=base_fontsize + 2)
        ax.set_xticks(TOPIC_COUNTS)
        ax.set_xlabel("So topic (k)", fontsize=base_fontsize)
        ax.set_ylabel(ylabel, fontsize=base_fontsize)
        ax.tick_params(labelsize=base_fontsize - 1)
        ax.spines[["top", "right"]].set_visible(False)

    handles_by_label = dict(zip(*axes.flat[0].get_legend_handles_labels()[::-1]))
    ordered_labels = [label_of[m] for m in models_to_draw]
    handles = [handles_by_label[label] for label in ordered_labels]
    ncol = 3 if len(models_to_draw) > 4 else len(models_to_draw)
    fig.legend(
        handles, ordered_labels, loc="upper center", ncol=ncol,
        bbox_to_anchor=(0.5, 1.07), frameon=False, fontsize=base_fontsize,
    )
    fig.suptitle(title, y=1.15, fontsize=base_fontsize + 4, fontweight="bold")
    fig.tight_layout(rect=(0, 0, 1, 0.90))
    fig.savefig(output_path, bbox_inches="tight", facecolor="white")
    plt.show()
    print(f"Da luu: {output_path}")
    return output_path

## 1. Coherence (WEC-in)

Cosine trung bình giữa các cặp top-term trong từng topic (Word2Vec huấn luyện trên chính corpus). Càng cao càng tốt.

In [ ]:
plot_metric_grid(
    frame,
    metric="wec_in",
    ylabel="WEC-in",
    title="Coherence (WEC-in) theo so topic, tren 4 corpus tieng Viet (CafeBERT)",
    output_path=OUTPUT_DIR / "topic_coherence.png",
)

## 2. Diversity

Tỉ lệ top-term không trùng lặp trên toàn bộ topic. Giá trị cao **không** đồng nghĩa coherence cao — đọc cùng biểu đồ coherence ở trên.

In [ ]:
plot_metric_grid(
    frame,
    metric="topic_diversity",
    ylabel="Topic diversity",
    title="Diversity theo so topic, tren 4 corpus tieng Viet (CafeBERT)",
    output_path=OUTPUT_DIR / "topic_diversity.png",
)

## 3. Runtime — fit-only vs pipeline (KHÔNG gộp lẫn)

- **`fit_seconds`** (fit-only, warm): chỉ thời gian fit mô hình sau khi embedding/CountVectorizer đã sẵn sàng — so sánh tốc độ thuật toán.
- **`pipeline_seconds`**: cộng thêm chi phí encode CafeBERT (S³/BERTopic) hoặc CountVectorizer (LDA/NMF) — không phải ablation encoder.

Cả hai ở thang log vì các phương pháp chênh nhau hàng chục/hàng trăm lần.

In [ ]:
plot_metric_grid(
    frame,
    metric="fit_seconds",
    ylabel="Fit-only, warm (giay, thang log)",
    title="Runtime fit-only theo so topic, tren 4 corpus tieng Viet (CafeBERT)",
    output_path=OUTPUT_DIR / "runtime_fit_only.png",
    log_scale=True,
    # Ca 3 bien the deu suy tu 1 lan fit/refit da cache (xem docstring
    # plot_metric_grid) nen fit_seconds gan nhu trung khop tuyet doi --
    # chi ve 1 duong dai dien, tranh nham la axial/angular "mat line".
    s3_models=["s3_combined"],
    s3_label_override="S³ (axial ≈ angular ≈ combined)",
)

In [ ]:
plot_metric_grid(
    frame,
    metric="pipeline_seconds",
    ylabel="Pipeline cold-reference (giay, thang log)",
    title="Runtime pipeline (bieu dien + fit) theo so topic, tren 4 corpus tieng Viet",
    output_path=OUTPUT_DIR / "runtime_pipeline.png",
    log_scale=True,
    s3_models=["s3_combined"],
    s3_label_override="S³ (axial ≈ angular ≈ combined)",
)

## 4. Robustness (C_NPMI)

Gensim C_NPMI — chỉ số coherence phụ (đồng xuất hiện từ), dùng kiểm tra không đồng thuận với WEC-in, **không** dùng chọn winner.

In [ ]:
plot_metric_grid(
    frame,
    metric="c_npmi",
    ylabel="C_NPMI (robustness, khong dung chon winner)",
    title="Robustness (C_NPMI) theo so topic, tren 4 corpus tieng Viet (CafeBERT)",
    output_path=OUTPUT_DIR / "robustness_c_npmi.png",
)

## Phụ lục A: số ô WEC-in mà một biến thể S³ thắng

In [ ]:
def count_wec_wins(frame: pd.DataFrame, seed_scope: list[int]) -> tuple[int, int, dict[str, int]]:
    scoped = frame.loc[frame["seed"].isin(seed_scope)]
    total = 0
    s3_wins = 0
    corpus_wins = {corpus: 0 for corpus in CORPUS_ORDER}
    for (corpus, _seed, _k), group in scoped.groupby(["corpus", "seed", "n_topics"], observed=True):
        max_score = group["wec_in"].max()
        winners = set(group.loc[group["wec_in"].eq(max_score), "model"])
        total += 1
        if any(model.startswith("s3_") for model in winners):
            s3_wins += 1
            corpus_wins[corpus] += 1
    return s3_wins, total, corpus_wins


s3_wins, total_cells, corpus_wins = count_wec_wins(frame, SEEDS)
print(f"S3 dan dau WEC-in trong {s3_wins}/{total_cells} o corpus x seed x k.")
for corpus in CORPUS_ORDER:
    print(f"  {CORPUS_LABELS[corpus]:20s}: {corpus_wins[corpus]}/20")

## Phụ lục B: Concept Compass — từ vựng quanh 2 trục ngữ nghĩa (BATTERY vs CAMERA, ViSFD)

Load một checkpoint S³ thật (`artifacts/turftopic/<dataset>/models/model_n<k>_*.joblib` + `word_embeddings.npy` cùng thư mục), chiếu **toàn bộ từ vựng** lên không gian trục ngữ nghĩa bằng `ica.transform(word_embeddings)` (tương đương $W = V C^{\top}$).

Dùng checkpoint **ViSFD** (không phải Vietnamese-news) vì ViSFD có nhãn khía cạnh thật (BATTERY, CAMERA...) để đối chiếu — chọn hai trục có AUC cao nhất với hai khía cạnh này (dò bằng `s3_reproduction.validate_visfd.correlate_axes`, xem `AXIS_X`/`AXIS_Y` bên dưới):

- `AXIS_X = 1` → **camera**: *chụp, selfie, camera, cam đẹp, máy ảnh, xóa phông…* (AUC≈0,80 với nhãn CAMERA)
- `AXIS_Y = 21` → **battery/pin**: *pin trâu, trâu bò…* (AUC≈0,68 với nhãn BATTERY)

**Nhãn trục cố ý đảo vị trí**: chữ dọc bên trái ghi thông tin trục hoành (`AXIS_X`), chữ ngang phía dưới ghi thông tin trục tung (`AXIS_Y`) — ngược quy ước mặc định của matplotlib theo yêu cầu trình bày.

**Chọn từ THEO TRỤC, không theo lưới toạ độ**: mỗi trục có 2 cực — "hưởng ứng" (giá trị lớn nhất) và "bài xích" (giá trị nhỏ/âm nhất) — lấy `N_PER_POLE` từ mỗi cực trong 4 cực (x+, x-, y+, y-), vị trí nhãn tự nhiên theo đúng phân bố dữ liệu (không chia ô đều — nhìn "ngăn nắp giả tạo"). **Không chấm điểm đánh dấu** — chữ viết thẳng tại toạ độ (nền trắng mờ phía sau khi 2 nhãn gần nhau). Cỡ chữ `BASE_FONTSIZE = 360`, figsize giữ nguyên `128×116` (lần trước gấp đôi cả font lẫn figsize cùng lúc nên tỉ lệ chữ/canvas không đổi — nhìn "vẫn nhỏ" khi scale về cùng độ rộng cố định trong paper/slide; lần này chỉ tăng font, giữ nguyên canvas, để chữ thật sự to hơn trong kết quả hiển thị). Đổi `DATASET`/`N_TOPICS`/`AXIS_X`/`AXIS_Y`/`GLOSS`/`N_PER_POLE`/`BASE_FONTSIZE` tuỳ ý. Cần chạy `make train-visfd` trước để có checkpoint.

In [ ]:
import glob
import json
import sys

import numpy as np

# checkpoint .joblib pickle class s3_reproduction.checkpoint.ModelCheckpoint --
# can co repo root tren sys.path de joblib.load() unpickle duoc.
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

DATASET = "visfd"            # co nhan khia canh that (BATTERY, CAMERA...) de doi chieu
N_TOPICS = 50                # phai co checkpoint model_n<N_TOPICS>_*.joblib
AXIS_X = 1                    # truc hoanh -- camera (AUC~0.80 voi nhan CAMERA)
AXIS_Y = 21                   # truc tung -- battery/pin (AUC~0.68 voi nhan BATTERY)
GLOSS = {1: "camera", 21: "battery (pin)"}   # chu giai ten truc tren nhan
BOX = (-4.0, 4.0)             # gioi han hien thi CA HAI truc -- cat bot cac tu cuc tri
                               # o xa (vd "pin trau" y~12.5) de tap trung vao vung dong
                               # tu vung nhat, giong pham vi Figure 7 cua paper goc.
MAX_LABELS = 40                # so tu duoc chon NGAU NHIEN (khong chia luoi) trong BOX --
                               # giong cach Figure 7 cua paper S3 goc trai tu tu nhien
                               # theo phan bo that cua du lieu, khong ep deu qua luoi.
RANDOM_SEED = 0                # seed co dinh de ket qua tai lap duoc giua cac lan chay.
BASE_FONTSIZE = 360            # gap 4 lan ban truoc (90), GIU NGUYEN figsize (128x116)
                               # de ty le chu/canvas thuc su tang (khong lap lai loi
                               # "gap doi ca font lan canvas" khien chu nhin khong doi).

ds_dir = ROOT / "artifacts" / "turftopic" / DATASET
cands = sorted(glob.glob(str(ds_dir / "models" / f"model_n{N_TOPICS}_*.joblib")))
if not cands:
    raise FileNotFoundError(
        f"Khong tim thay checkpoint model_n{N_TOPICS}_*.joblib trong {ds_dir/'models'}. "
        f"Chay 'make train-visfd' truoc."
    )
import joblib

checkpoint = joblib.load(cands[-1])
vocab = json.loads((ds_dir / "vocabulary.json").read_text(encoding="utf-8"))
word_emb = np.load(ds_dir / "word_embeddings.npy")
assert list(checkpoint.vocabulary) == list(vocab), "vocabulary trong checkpoint khac file vocabulary.json"
assert word_emb.shape[0] == len(vocab), "word_embeddings.npy khong khop so tu vung"

# ica.transform lo center + whiten + unmix -- toa do tung tu tren k truc (tuong duong W = V C^T)
coords = checkpoint.ica.transform(word_emb)   # (n_words, k)
print(f"checkpoint: {Path(cands[-1]).name}  |  k={coords.shape[1]}  |  {len(vocab)} tu vung")
print(f"Truc hoanh = Topic {AXIS_X} ({GLOSS[AXIS_X]}), truc tung = Topic {AXIS_Y} ({GLOSS[AXIS_Y]})")

In [ ]:
def concept_compass(
    coords: np.ndarray,
    vocab: list[str],
    axis_x: int,
    axis_y: int,
    gloss: dict[int, str],
    output_path: Path,
    box: tuple[float, float] = (-4.0, 4.0),
    max_labels: int = 40,
    random_seed: int = 0,
    base_fontsize: int = 360,
) -> pd.DataFrame:
    """Chon tu NGAU NHIEN trong [box] (giong Figure 7 cua paper S3 goc):
    KHONG chia luoi, KHONG ep vi tri deu -- lay max_labels tu bang random
    sample (seed co dinh de tai lap duoc) tu tap hop tat ca tu nam trong
    [box]. Vi ban than phan bo tu vung da tu nhien day dac gan tam va thua
    dan ra bien, random sample tu do van cho ra mat do giong that (giong
    Figure 7), khong "ngan nap gia tao" nhu chia luoi truoc day.
    KHONG ve scatter nen, KHONG ve duong luoi nen (chi con truc x=0/y=0) --
    nen trang sach.
    Nhan truc co tinh CO Y (dao nguoc quy uoc matplotlib): chu doc (vi tri Y
    mac dinh) ghi thong tin truc HOANH (axis_x); chu ngang (vi tri X mac
    dinh) ghi thong tin truc TUNG (axis_y).
    Tra ve DataFrame [word, x, y] cua cac tu duoc ghi nhan."""
    x_full = coords[:, axis_x]
    y_full = coords[:, axis_y]
    in_box = (x_full >= box[0]) & (x_full <= box[1]) & (y_full >= box[0]) & (y_full <= box[1])
    idx_in_box = np.where(in_box)[0]

    rng = np.random.default_rng(random_seed)
    n_pick = min(max_labels, len(idx_in_box))
    picks = sorted(rng.choice(idx_in_box, size=n_pick, replace=False).tolist())
    print(f"labels: {len(picks)} (random trong box {box}, tu {len(idx_in_box)} tu ung vien)")

    plt.style.use("default")
    plt.rcParams.update({"font.size": base_fontsize})
    fig, ax = plt.subplots(figsize=(128, 116), dpi=80)

    for i in picks:
        ax.annotate(
            vocab[i], (x_full[i], y_full[i]), ha="center", va="center",
            fontsize=base_fontsize, zorder=5, color="#0f172a",
        )

    ax.axhline(0, color="#334155", linewidth=3, zorder=2)
    ax.axvline(0, color="#334155", linewidth=3, zorder=2)
    ax.set_xlim(*box)
    ax.set_ylim(*box)
    ax.grid(False)
    for spine in ("top", "right"):
        ax.spines[spine].set_visible(False)
    # DAO nhan: vi tri truc TUNG hien thong tin truc HOANH va nguoc lai.
    ax.set_ylabel(f"Topic {axis_x} — {gloss.get(axis_x, '')}", fontsize=base_fontsize + 40, fontweight="bold")
    ax.set_xlabel(f"Topic {axis_y} — {gloss.get(axis_y, '')}", fontsize=base_fontsize + 40, fontweight="bold")
    ax.set_title(
        f"Từ vựng nổi bật nhất trên hai trục ngữ nghĩa S³ ({DATASET}, k={coords.shape[1]})",
        fontsize=base_fontsize, fontweight="bold", pad=30,
    )
    ax.tick_params(labelsize=base_fontsize - 40)
    fig.tight_layout()
    fig.savefig(output_path, bbox_inches="tight", facecolor="white")
    plt.show()
    print(f"Da luu: {output_path}  ({len(picks)} tu duoc ghi nhan)")

    picks_sorted = sorted(picks, key=lambda i: (x_full[i], y_full[i]))
    return (
        pd.DataFrame({"word": [vocab[i] for i in picks_sorted],
                      f"x_topic{axis_x}": np.round(x_full[picks_sorted], 3),
                      f"y_topic{axis_y}": np.round(y_full[picks_sorted], 3)})
        .reset_index(drop=True)
    )


compass_df = concept_compass(
    coords, vocab, AXIS_X, AXIS_Y, GLOSS,
    output_path=OUTPUT_DIR / f"concept_compass_{DATASET}_t{AXIS_X}_vs_t{AXIS_Y}.png",
    box=BOX, max_labels=MAX_LABELS, random_seed=RANDOM_SEED, base_fontsize=BASE_FONTSIZE,
)
compass_df

In [ ]:
# In vi tri hinh chieu -- top 15 tu tren tung cuc (+/-) cua moi truc.
def top_words_on_axis(coords, vocab, axis, n=15):
    v = coords[:, axis]
    pos = np.argsort(-v)[:n]
    neg = np.argsort(v)[:n]
    print(f"--- Topic {axis}: cuc DUONG (x lon nhat) ---")
    for i in pos:
        print(f"   {v[i]:+7.3f}  {vocab[i]}")
    print(f"--- Topic {axis}: cuc AM (x nho nhat) ---")
    for i in neg:
        print(f"   {v[i]:+7.3f}  {vocab[i]}")


top_words_on_axis(coords, vocab, AXIS_X)
print()
top_words_on_axis(coords, vocab, AXIS_Y)

## Dùng biểu đồ trong `report/paper.tex`

Copy PNG vào `report/figures/` với tên riêng của nhóm (không đè lên ảnh tái hiện từ paper gốc):
```
cp benchmark/cafebert_full/notebook_charts/topic_coherence.png   report/figures/vn_benchmark_coherence.png
cp benchmark/cafebert_full/notebook_charts/runtime_fit_only.png  report/figures/vn_benchmark_runtime.png
cp benchmark/cafebert_full/notebook_charts/concept_compass_visfd_t1_vs_t21.png  report/figures/concept_compass.png
```